In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd, numpy as np
from IPython.display import Markdown, Image, display
P = ROOT / "data" / "processed"
F = ROOT / "docs" / "figures"
pd.options.display.float_format = "{:,.4f}".format
def table(name): return pd.read_csv(P / name)
def summary(name): return json.loads((P / name).read_text())
def fig(name): display(Image(filename=str(F / name)))
print("Simulated data. Every number below is read from executed pipeline outputs.")


Simulated data. Every number below is read from executed pipeline outputs.


# M1. Simulated quote market (Simulated data)

Every row is generated by `retailpricing.data_gen` from illustrative assumptions in `config/calibration.yaml`. Each parameter carries a status and a source note. Nothing here is real insurer experience.

Our price loads on an unobserved latent intent `u`, which also raises conversion. That is the endogeneity built in on purpose so M3 and M4 have something real to demonstrate.

In [2]:
s = summary("m1_summary.json")
pd.Series(s).to_frame("value")

,value
data_label,Simulated data
rows,200000
new_business_rows,140108
renewal_rows,59892
randomised_test_rows,20905
nb_conversion_rate,0.0920
renewal_rate,0.7833
avg_nb_premium,641.8437
avg_our_rank,4.5604
quality_checks_passed,31


## Data quality checks

In [3]:
dq = table("data_quality_report.csv")
display(Markdown(f"{int(dq['passed'].sum())} of {len(dq)} checks passed."))
dq

31 of 31 checks passed.

,check,passed,detail
0,required columns present,True,19 required
1,"row count within 150,000 to 250,000",True,"200,000 rows"
2,quote_id unique,True,NaN
3,customer_id unique,True,NaN
4,no nulls in required fields,True,NaN
5,our_premium positive,True,NaN
6,competitor prices sorted ascending,True,NaN
7,comp_min <= comp_median <= comp_max,True,NaN
8,comp_mean between min and max,True,NaN
9,rank recomputes from competitor prices,True,NaN


## Configuration status

In [4]:
cs = table("config_status_table.csv")
display(cs.groupby("status").size().to_frame("parameters"))
cs.head(15)

,parameters
status,
illustrative_assumption,95


,parameter,status,source_note
0,volumes.n_quotes,illustrative_assumption,NaN
1,volumes.start_date,illustrative_assumption,NaN
2,volumes.n_months,illustrative_assumption,NaN
3,volumes.new_business_share,illustrative_assumption,NaN
4,volumes.volume_seasonality_amplitude,illustrative_assumption,NaN
5,volumes.age_band_mix,illustrative_assumption,NaN
6,volumes.region_mix,illustrative_assumption,NaN
7,volumes.risk_band_mix,illustrative_assumption,NaN
8,volumes.channel_mix_new_business,illustrative_assumption,NaN
9,volumes.channel_mix_renewal,illustrative_assumption,NaN


## Reproducibility
The SHA-256 of the frame is stored beside the parquet file. Two runs with the same seed must give the same hash (tested in `tests/test_generator.py`).

In [5]:
print((ROOT / "data" / "raw" / "simulated_quotes.sha256").read_text())

165213e22e28bcefbecb9312d39f782ef975bcd24caf7fc173ad926e5f4fab8f

